# Lightning Data Module
This notebook preprocesses CSV data for each of train, val and test into three prepcrossed
NPZ folders. These are in turn loaded into the pytorch lightning datamodule. 


In [1]:
from pathlib import Path
import numpy as np, pandas as pd, json, os
from tqdm import tqdm

# ---- PATHS to Train, Val and Test directories ===
TRAIN_DIR = Path("data/forecasting_train_v1.1/train/data/")
VAL_DIR   = Path("data/forecasting_val_v1.1/val/data/")
TEST_DIR  = Path("data/forecasting_test_v1.1/test_obs/data/")

# Where to write the preprocessed files (one .npz per scene)
OUT_ROOT = Path("preprocessed/agent_only_npz")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Observation / prediction horizons
OBS_LEN, PRED_LEN = 20, 30  # 2s past, 3s future @10Hz

# Preprocessing settings
FRACTION = 0.10        # process 10% of each split
SEED = 42              # change to add another random 10% later
UNCOMPRESSED = True    # np.savez (faster) vs np.savez_compressed (smaller)


In [2]:
def _load_agent_xy(csv_path: Path) -> np.ndarray:
    """
    Return XY positions for the AGENT track in one scene (float32).
    """
    df = pd.read_csv(
        csv_path,
        usecols=["TIMESTAMP","TRACK_ID","OBJECT_TYPE","X","Y"],
        engine="c",
    ).sort_values(["TIMESTAMP","TRACK_ID"])
    agent = df.loc[df["OBJECT_TYPE"]=="AGENT","TRACK_ID"].iloc[0]
    a = df[df["TRACK_ID"]==agent].sort_values("TIMESTAMP")
    return a[["X","Y"]].to_numpy(np.float32)

def preprocess_split(split_name: str, in_dir: Path, out_root: Path,
                     fraction=FRACTION, seed=SEED, uncompressed=UNCOMPRESSED):
    """
    Process ~fraction of scenes from in_dir into out_root/split_name/*.npz
    Resumable: skips files that already exist and keeps a manifest.json.
    """
    if not in_dir.exists():
        print(f"[{split_name}] SKIP — input folder not found: {in_dir.resolve()}")
        return

    out_dir = out_root / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = out_dir / "manifest.json"

    # load/update manifest (resume)
    manifest = {"obs_len": OBS_LEN, "pred_len": PRED_LEN, "source_dir": str(in_dir), "files": []}
    if manifest_path.exists():
        try:
            manifest.update(json.loads(manifest_path.read_text()))
        except Exception:
            pass

    csv_files = sorted(in_dir.glob("*.csv"))
    done = {Path(f).stem for f in manifest.get("files", [])}
    pending = [p for p in csv_files if not (out_dir / f"{p.stem}.npz").exists() and p.stem not in done]

    if not pending:
        print(f"[{split_name}] Nothing to do. ({len(csv_files)} total, {len(done)} already done)")
        return

    # choose ~fraction of pending (deterministic random)
    rng = np.random.default_rng(seed)
    k = max(1, int(np.ceil(fraction * len(pending))))
    idx = sorted(rng.choice(len(pending), size=k, replace=False))
    to_process = [pending[i] for i in idx]

    print(f"[{split_name}] Processing {len(to_process)}/{len(csv_files)} scenes (~{fraction*100:.0f}%) → {out_dir}")
    pbar = tqdm(total=len(to_process), desc=f"{split_name} export", unit="scene")
    processed, skipped, errors = 0, 0, 0

    for p in to_process:
        try:
            xy = _load_agent_xy(p)
            if xy.shape[0] < OBS_LEN + PRED_LEN:
                skipped += 1
            else:
                origin = xy[OBS_LEN-1]
                past   = xy[:OBS_LEN] - origin
                future = xy[OBS_LEN:OBS_LEN+PRED_LEN] - origin
                out_path = out_dir / f"{p.stem}.npz"
                if uncompressed:
                    np.savez(out_path, past=past, future=future, origin=origin.astype(np.float32), path=p.name)
                else:
                    np.savez_compressed(out_path, past=past, future=future, origin=origin.astype(np.float32), path=p.name)
                manifest["files"].append(p.name)
                processed += 1
        except Exception as e:
            errors += 1
            print(f"[{split_name}][WARN] {p.name}: {e}")
        finally:
            pbar.update(1)

    pbar.close()
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f"[{split_name}] Done. OK={processed} Skipped={skipped} Errors={errors}. Out: {out_dir}")


In [3]:
from pathlib import Path
import numpy as np, pandas as pd, json
from tqdm import tqdm

OBS_LEN, PRED_LEN = 20, 30
TEST_DIR = Path("data/forecasting_test_v1.1/test_obs/data/")
OUT_ROOT = Path("preprocessed/agent_only_npz"); OUT_ROOT.mkdir(parents=True, exist_ok=True)
FRACTION = 0.10
SEED = 42
UNCOMPRESSED = True

def _load_agent_xy(csv_path: Path) -> np.ndarray:
    df = pd.read_csv(
        csv_path,
        usecols=["TIMESTAMP","TRACK_ID","OBJECT_TYPE","X","Y"],
        engine="c",
    ).sort_values(["TIMESTAMP","TRACK_ID"])
    agent = df.loc[df["OBJECT_TYPE"]=="AGENT","TRACK_ID"].iloc[0]
    a = df[df["TRACK_ID"]==agent].sort_values("TIMESTAMP")
    return a[["X","Y"]].to_numpy(np.float32)

def preprocess_test_allow_nofuture(in_dir: Path, out_root: Path, fraction=0.10, seed=42, uncompressed=True):
    split_name = "test"
    if not in_dir.exists():
        print(f"[{split_name}] SKIP — input folder not found: {in_dir.resolve()}")
        return
    out_dir = out_root / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = out_dir / "manifest.json"

    # resume
    manifest = {"obs_len": OBS_LEN, "pred_len": PRED_LEN, "source_dir": str(in_dir), "files": [], "notes":"future may be missing"}
    if manifest_path.exists():
        try:
            manifest.update(json.loads(manifest_path.read_text()))
        except Exception:
            pass

    csv_files = sorted(in_dir.glob("*.csv"))
    done = {Path(f).stem for f in manifest.get("files", [])}
    pending = [p for p in csv_files if not (out_dir / f"{p.stem}.npz").exists() and p.stem not in done]

    # take ~fraction (deterministic)
    rng = np.random.default_rng(seed)
    k = max(1, int(np.ceil(fraction * len(pending))))
    idx = sorted(rng.choice(len(pending), size=k, replace=False))
    to_process = [pending[i] for i in idx]

    print(f"[{split_name}] Processing {len(to_process)}/{len(csv_files)} scenes (~{fraction*100:.0f}%) → {out_dir}")
    pbar = tqdm(total=len(to_process), desc=f"{split_name} export", unit="scene")
    ok = skip_too_short = err = 0

    for p in to_process:
        try:
            xy = _load_agent_xy(p)
            if xy.shape[0] < OBS_LEN:  # cannot build even past
                skip_too_short += 1
            else:
                # center at last observed (or last available if shorter than OBS_LEN)
                cut = min(OBS_LEN, xy.shape[0])
                origin = xy[cut-1]
                past = np.zeros((OBS_LEN, 2), np.float32)
                past[:cut] = xy[:cut] - origin  # pad leading zeros if cut<OBS_LEN

                has_future = int(xy.shape[0] >= OBS_LEN + PRED_LEN)
                if has_future:
                    fut = xy[OBS_LEN:OBS_LEN+PRED_LEN] - origin
                else:
                    fut = np.full((PRED_LEN, 2), np.nan, dtype=np.float32)

                out_path = out_dir / f"{p.stem}.npz"
                if uncompressed:
                    np.savez(out_path, past=past, future=fut, origin=origin.astype(np.float32),
                             path=p.name, has_future=np.array(has_future, dtype=np.int8))
                else:
                    np.savez_compressed(out_path, past=past, future=fut, origin=origin.astype(np.float32),
                                        path=p.name, has_future=np.array(has_future, dtype=np.int8))
                manifest["files"].append(p.name)
                ok += 1
        except Exception as e:
            err += 1
            print(f"[{split_name}][ERR] {p.name}: {e}")
        finally:
            pbar.update(1)

    pbar.close()
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f"[{split_name}] Done. OK={ok} Short={skip_too_short} Err={err}. Out: {out_dir}")


In [4]:
#preprocess_split("train", TRAIN_DIR, OUT_ROOT, FRACTION, SEED, UNCOMPRESSED)
#preprocess_split("val",   VAL_DIR,   OUT_ROOT, FRACTION, SEED, UNCOMPRESSED)
#preprocess_test_allow_nofuture(TEST_DIR, OUT_ROOT, FRACTION, SEED, UNCOMPRESSED)


In [5]:
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset

PRED_LEN = 30  # only used to make a placeholder if you ever want one

class AgentNPZDataset(Dataset):
    """
    Loads per-scene NPZ files written by your preprocessor.
    - Train/Val: include_future=True  -> returns 'future'
    - Test:      include_future=False -> omits 'future' (clean inference)
    Always returns: 'past', 'origin', 'path', and 'has_future' (bool)
    """
    def __init__(self, npz_dir: Path, include_future: bool = True):
        self.dir = Path(npz_dir)
        self.files = sorted(self.dir.glob("*.npz"))
        assert self.files, f"No .npz files in {self.dir}"
        self.include_future = include_future

    def __len__(self): 
        return len(self.files)

    def __getitem__(self, i):
        f = self.files[i]
        with np.load(f, mmap_mode="r") as z:
            past   = torch.from_numpy(z["past"])     # (20,2)
            origin = torch.from_numpy(z["origin"])   # (2,)
            path   = str(z["path"])
            has_future = bool(int(z["has_future"])) if "has_future" in z.files else True

            sample = {"past": past, "origin": origin, "path": path, "has_future": torch.tensor(has_future)}

            if self.include_future:
                # For train/val we include future; if a test file sneaks in with no future,
                # we still return a tensor (zeros) so batch collation works.
                if has_future and "future" in z.files:
                    fut = torch.from_numpy(z["future"])  # (30,2)
                else:
                    fut = torch.zeros((PRED_LEN, 2), dtype=past.dtype)
                sample["future"] = fut

            return sample


In [6]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader

class AgentNPZDataModule(pl.LightningDataModule):
    def __init__(self, root_dir="preprocessed/agent_only_npz", batch_size=256, num_workers=4,
                 include_future_in_test: bool = False):
        super().__init__()
        self.root_dir = Path(root_dir)
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.include_future_in_test = include_future_in_test
        self.splits = {}
        self._train_dl = self._val_dl = self._test_dl = None

    def setup(self, stage=None):
        for split in ("train","val","test"):
            split_dir = self.root_dir / split
            if split_dir.exists() and list(split_dir.glob("*.npz")):
                include_future = not (split == "test" and not self.include_future_in_test)
                self.splits[split] = AgentNPZDataset(split_dir, include_future=include_future)
            else:
                print(f"[DataModule] Split '{split}' not available at {split_dir} — skipping.")

        # build and cache the loaders ONCE so workers stay alive across calls
        def _dl(ds, shuffle):
            return DataLoader(
                ds, batch_size=self.batch_size, shuffle=shuffle,
                num_workers=self.num_workers, pin_memory=True,
                persistent_workers=self.num_workers>0, prefetch_factor=2
            )

        if "train" in self.splits and self._train_dl is None:
            self._train_dl = _dl(self.splits["train"], True)
        if "val" in self.splits and self._val_dl is None:
            self._val_dl = _dl(self.splits["val"], False)
        if "test" in self.splits and self._test_dl is None:
            self._test_dl = _dl(self.splits["test"], False)

    def train_dataloader(self): return self._train_dl
    def val_dataloader(self):   return self._val_dl
    def test_dataloader(self):  return self._test_dl


c:\Users\fiona\Documents\GitHub\Transformers\transformers-mini\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import os

dm = AgentNPZDataModule(
    root_dir="preprocessed/agent_only_npz",
    batch_size=512,
    num_workers=max(1, (os.cpu_count() or 2)//2),
    include_future_in_test=False  # test batches won't include 'future'
)
dm.setup()



In [10]:
# Progress peek without spawning workers (fast first batch)
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
import math

# use the already-built splits from your dm
assert "train" in dm.splits, "No train split available"

BATCH = dm.batch_size
N = len(dm.splits["train"])
dl_quick = DataLoader(dm.splits["train"], batch_size=BATCH, shuffle=False, num_workers=0)

for i, b in enumerate(tqdm(dl_quick, total=min(20, math.ceil(N/BATCH)), desc="Peeking train")):
    if i == 0:
        keys = list(b.keys())
        shapes = {k: tuple(v.shape) for k, v in b.items() if hasattr(v, "shape")}
        print(f"[train] keys={keys}")
        print(f"[train] shapes={shapes}")
        if "has_future" in b:
            print(f"[train] has_future sum: {int(b['has_future'].sum())}/{b['has_future'].numel()}")
    if i+1 >= 20:  # stop after ~20 batches
        break


Peeking train:   5%|▌         | 1/20 [00:04<01:33,  4.94s/it]

[train] keys=['past', 'origin', 'path', 'has_future', 'future']
[train] shapes={'past': (512, 20, 2), 'origin': (512, 2), 'has_future': (512,), 'future': (512, 30, 2)}
[train] has_future sum: 512/512


Peeking train:  55%|█████▌    | 11/20 [01:01<00:50,  5.58s/it]


KeyboardInterrupt: 